# Resumen Intervencional: Beta=0 vs Beta óptima

Igual que en `Resumen_Observacional.ipynb`, pero sobre los 8 experimentos **Intervencional** (4 ruidos
aditivos + 4 multiplicativos): compara el modelo base (`Beta=0`) frente al modelo con la `Beta`
óptima, y comprueba con Wilcoxon pareado si las mejoras son significativas.

**La Beta óptima no se re-selecciona aquí.** Se importa, por (Experimento, N), del resultado ya
calculado en `Resumen_Observacional.ipynb` (columna `beta_optima` de
`tablas/resumen_observacional_beta_n{50,100}.csv`), no de un nuevo criterio basado en las métricas
interventional (RF Acc/MMD/MAE Z). Como el análisis observacional no tiene eje `Y_do`, la misma Beta
se usa para `do(Y)=0.0`, `do(Y)=1.0` y su media — solo cambia entre N=50 y N=100.

Diferencias frente al caso observacional:
- **No se incluye `HSIC(Z,Y)`**: bajo intervención, Y se fija externamente (`do(Y)`) y ya no se genera
  por su ecuación estructural, así que probar la independencia del residuo de Z frente a Y no tiene
  sentido (de hecho la columna `HSIC(Z,Y)` de estos CSVs viene vacía). Las métricas usadas son
  `MAE Z`, `HSIC(Z,X)` y `RF Acc`.
- **Nueva dimensión `Y_do`**: cada fila corresponde a `do(Y)=0.0` o `do(Y)=1.0`. Se calcula todo por
  separado para cada intervención, y además una versión "media" que agrupa ambas.
- Los 4 CSVs `multiplicativo_*` solo tienen 10 seeds (frente a 20 en `aditivo_*`), así que sus tests
  de Wilcoxon tienen menos potencia.

Todo el análisis se repite para `N=50` y `N=100`.

In [1]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

NOTEBOOKS_DIR = Path.cwd().parent if Path.cwd().name == "Resumenes" else Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent
TABLAS_DIR = NOTEBOOKS_DIR / "Resumenes" / "tablas"

ALPHA = 0.05
METRICAS = ["MAE Z", "HSIC(Z,X)", "RF Acc"]  # menor es mejor en las 3; sin HSIC(Z,Y) (no aplica bajo do(Y))

## Definición de los 8 experimentos intervencionales

In [2]:
EXPERIMENTOS = {
    "Aditivo Gaussiano": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_gausian.csv",
    "Aditivo Exponencial": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_exponencial.csv",
    "Aditivo Gamma": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_gamma.csv",
    "Aditivo Uniforme": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_uniforme.csv",
    "Multiplicativo Gaussiano": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_gausian.csv",
    "Multiplicativo Exponencial": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_exponencial.csv",
    "Multiplicativo Gamma": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_gamma.csv",
    "Multiplicativo Uniforme": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_uniforme.csv",
}

for nombre, path in EXPERIMENTOS.items():
    assert path.exists(), f"No existe: {path}"


## Carga de la Beta óptima (importada de Resumen_Observacional)

In [3]:
def cargar_beta_observacional():
    """Carga la Beta óptima por (Experimento, N) desde los CSVs de Resumen_Observacional.

    Devuelve un dict {(Experimento, N): beta_optima}.
    """
    paths_por_n = {
        50: TABLAS_DIR / "resumen_observacional_beta_n50.csv",
        100: TABLAS_DIR / "resumen_observacional_beta_n100.csv",
    }
    mapa = {}
    for n_filter, path in paths_por_n.items():
        assert path.exists(), f"No existe: {path}"
        df_beta = pd.read_csv(path)
        for _, fila in df_beta.iterrows():
            mapa[(fila["Experimento"], n_filter)] = float(fila["beta_optima"])
    return mapa


BETA_OPTIMA_OBSERVACIONAL = cargar_beta_observacional()

## Función de cálculo de valores en Beta=0 y Beta=beta_optima

In [4]:
def valores_para_beta(df: pd.DataFrame, beta_opt: float, n_filter: int, y_do_filter, metrics=METRICAS):
    """Calcula baseline (Beta=0) y valores en Beta=beta_opt, promediando por seed(s).

    `beta_opt` se recibe ya decidido (importado de Resumen_Observacional): esta función no
    selecciona ninguna Beta, solo agrega las métricas de `df` en Beta=0 y en Beta=beta_opt.
    `y_do_filter`: 0.0 o 1.0 para quedarse con esa intervención, o None para agrupar ambas (media).

    Devuelve (baseline: Series, valores_opt: Series).
    """
    df_n = df[df["N"] == n_filter]
    if y_do_filter is not None:
        df_n = df_n[df_n["Y_do"] == y_do_filter]

    media_por_beta = df_n.groupby("Beta")[list(metrics)].mean()
    baseline = media_por_beta.loc[0.0]
    valores_opt = media_por_beta.loc[beta_opt]
    return baseline, valores_opt


## Función de test de Wilcoxon pareado

Se empareja por `(Seed, Y_do)` en vez de solo `Seed`: así funciona igual si se filtra a un único
`Y_do` (queda un valor de `Y_do` por seed, como antes) o si se agrupan ambos (quedan 2 filas por seed,
una por intervención, cada una emparejada con su contraparte en la misma condición).


In [5]:
def wilcoxon_experimento(df: pd.DataFrame, beta_opt: float, n_filter: int, y_do_filter, metrics=METRICAS):
    """Wilcoxon signed-rank pareado entre Beta=0 y Beta=beta_opt, por métrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_opt es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    """
    df_n = df[df["N"] == n_filter]
    if y_do_filter is not None:
        df_n = df_n[df_n["Y_do"] == y_do_filter]

    base = df_n[df_n["Beta"] == 0.0].set_index(["Seed", "Y_do"])[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index(["Seed", "Y_do"])[list(metrics)]
    pares_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[pares_comunes]
    opt = opt.loc[pares_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(pares_comunes)


## Cálculo (tabla resumen + p-valores) para un (N, do(Y)) dado

In [6]:
def analizar(n_filter: int, y_do_filter):
    """Ejecuta valores_para_beta + wilcoxon_experimento para los 8 experimentos, a N y Y_do fijos.

    y_do_filter: 0.0, 1.0, o None (media entre ambas intervenciones). La Beta usada para cada
    experimento se toma de BETA_OPTIMA_OBSERVACIONAL, no se selecciona aquí.
    Devuelve (resumen: DataFrame, p_values: DataFrame).
    """
    filas_resumen = []
    filas_p = []

    for nombre, path in EXPERIMENTOS.items():
        df = pd.read_csv(path)
        beta_opt = BETA_OPTIMA_OBSERVACIONAL[(nombre, n_filter)]
        baseline, valores_opt = valores_para_beta(df, beta_opt, n_filter, y_do_filter)

        filas_resumen.append({
            "Experimento": nombre,
            "beta_optima": beta_opt,
            "MAE(Z) beta=0": baseline["MAE Z"],
            "MAE(Z) beta_optima": valores_opt["MAE Z"],
            "HSIC(Z,X) beta=0": baseline["HSIC(Z,X)"],
            "HSIC(Z,X) beta_optima": valores_opt["HSIC(Z,X)"],
            "RF Acc beta=0": baseline["RF Acc"],
            "RF Acc beta_optima": valores_opt["RF Acc"],
            "Criterio": f"Beta importada de Resumen_Observacional (N={n_filter})",
        })

        p_valores, n_pares = wilcoxon_experimento(df, beta_opt, n_filter, y_do_filter)
        filas_p.append({"Experimento": nombre, **p_valores, "n_pares": n_pares})

    resumen = pd.DataFrame(filas_resumen)
    p_values = pd.DataFrame(filas_p).set_index("Experimento")
    return resumen, p_values


resultados = {}
for n_filter in (50, 100):
    for y_do_filter, etiqueta in ((0.0, "ydo0"), (1.0, "ydo1"), (None, "media")):
        resultados[(n_filter, etiqueta)] = analizar(n_filter, y_do_filter)


## Funciones de formato y resaltado en negrita

In [7]:
COLUMNA_A_METRICA = {
    "MAE(Z) beta_optima": "MAE Z",
    "HSIC(Z,X) beta_optima": "HSIC(Z,X)",
    "RF Acc beta_optima": "RF Acc",
}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("Experimento", "beta_optima", "Criterio")]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame, p_values: pd.DataFrame):
    """Tabla resumen con las celdas 'beta_optima' en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualización del notebook (un CSV plano no admite negrita).
    """
    fmt = formatear(resumen)

    def resaltar(row):
        p_exp = p_values.loc[row["Experimento"]]
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(p_exp[metrica]) and p_exp[metrica] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)


## N = 50

### do(Y) = 0.0

In [8]:
resumen, p_values = resultados[(50, "ydo0")][0], resultados[(50, "ydo0")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.900000,1.021620,1.250480,0.088420,0.136310,0.586250,0.607500,Beta importada de Resumen_Observacional (N=50)
1,Aditivo Exponencial,0.900000,0.984260,1.148020,0.122020,0.112860,0.666250,0.681250,Beta importada de Resumen_Observacional (N=50)
2,Aditivo Gamma,0.900000,1.009840,1.273010,0.222560,0.307420,0.750000,0.773750,Beta importada de Resumen_Observacional (N=50)
3,Aditivo Uniforme,0.300000,0.966670,0.997280,0.073490,0.071990,0.627500,0.622500,Beta importada de Resumen_Observacional (N=50)
4,Multiplicativo Gaussiano,0.400000,1.063700,1.028980,0.207960,0.184640,0.672500,0.667500,Beta importada de Resumen_Observacional (N=50)
5,Multiplicativo Exponencial,0.100000,0.752580,0.727380,0.212770,0.173140,0.700000,0.675000,Beta importada de Resumen_Observacional (N=50)
6,Multiplicativo Gamma,0.600000,0.826100,0.691170,0.244680,0.184780,0.702500,0.645000,Beta importada de Resumen_Observacional (N=50)
7,Multiplicativo Uniforme,0.400000,0.923970,0.916360,0.098640,0.080640,0.570000,0.580000,Beta importada de Resumen_Observacional (N=50)


In [9]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,1.0000,0.9958,0.8883,20
Aditivo Exponencial,0.9979,0.0220,0.8668,20
Aditivo Gamma,0.9995,0.9988,0.9109,20
Aditivo Uniforme,0.8529,0.1471,0.3150,20
Multiplicativo Gaussiano,0.0322,0.1875,0.3906,10
Multiplicativo Exponencial,0.0654,0.0967,0.2891,10
Multiplicativo Gamma,0.0322,0.1377,0.0039,10
Multiplicativo Uniforme,0.0244,0.0137,0.7500,10


### do(Y) = 1.0

In [10]:
resumen, p_values = resultados[(50, "ydo1")][0], resultados[(50, "ydo1")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.900000,0.921630,1.009210,0.062410,0.117050,0.582500,0.608750,Beta importada de Resumen_Observacional (N=50)
1,Aditivo Exponencial,0.900000,0.814900,0.977790,0.102290,0.090920,0.665000,0.677500,Beta importada de Resumen_Observacional (N=50)
2,Aditivo Gamma,0.900000,0.851990,0.991640,0.171350,0.250990,0.713750,0.742500,Beta importada de Resumen_Observacional (N=50)
3,Aditivo Uniforme,0.300000,0.925750,0.935120,0.048810,0.049880,0.607500,0.613750,Beta importada de Resumen_Observacional (N=50)
4,Multiplicativo Gaussiano,0.400000,0.712610,0.678740,0.352430,0.303470,0.660000,0.682500,Beta importada de Resumen_Observacional (N=50)
5,Multiplicativo Exponencial,0.100000,0.538480,0.487150,0.354820,0.276340,0.760000,0.720000,Beta importada de Resumen_Observacional (N=50)
6,Multiplicativo Gamma,0.600000,0.580750,0.469350,0.406940,0.266710,0.755000,0.732500,Beta importada de Resumen_Observacional (N=50)
7,Multiplicativo Uniforme,0.400000,0.528090,0.524840,0.154660,0.144450,0.567500,0.560000,Beta importada de Resumen_Observacional (N=50)


In [11]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,1.0000,0.9994,0.9374,20
Aditivo Exponencial,0.9999,0.0242,0.8091,20
Aditivo Gamma,0.9986,0.9940,0.9446,20
Aditivo Uniforme,0.9430,0.4702,0.7842,20
Multiplicativo Gaussiano,0.3125,0.2158,0.8281,10
Multiplicativo Exponencial,0.0527,0.0420,0.0547,10
Multiplicativo Gamma,0.0068,0.0049,0.2383,10
Multiplicativo Uniforme,0.1875,0.2158,0.4219,10


### Media entre do(Y)=0.0 y do(Y)=1.0

In [12]:
resumen, p_values = resultados[(50, "media")][0], resultados[(50, "media")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.900000,0.971630,1.129850,0.075420,0.126680,0.584380,0.608120,Beta importada de Resumen_Observacional (N=50)
1,Aditivo Exponencial,0.900000,0.899580,1.062900,0.112160,0.101890,0.665620,0.679380,Beta importada de Resumen_Observacional (N=50)
2,Aditivo Gamma,0.900000,0.930920,1.132320,0.196960,0.279200,0.731880,0.758120,Beta importada de Resumen_Observacional (N=50)
3,Aditivo Uniforme,0.300000,0.946210,0.966200,0.061150,0.060940,0.617500,0.618120,Beta importada de Resumen_Observacional (N=50)
4,Multiplicativo Gaussiano,0.400000,0.888160,0.853860,0.280190,0.244060,0.666250,0.675000,Beta importada de Resumen_Observacional (N=50)
5,Multiplicativo Exponencial,0.100000,0.645530,0.607270,0.283790,0.224740,0.730000,0.697500,Beta importada de Resumen_Observacional (N=50)
6,Multiplicativo Gamma,0.600000,0.703430,0.580260,0.325810,0.225740,0.728750,0.688750,Beta importada de Resumen_Observacional (N=50)
7,Multiplicativo Uniforme,0.400000,0.726030,0.720600,0.126650,0.112550,0.568750,0.570000,Beta importada de Resumen_Observacional (N=50)


In [13]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,1.0000,1.0000,0.9670,40
Aditivo Exponencial,1.0000,0.0022,0.9020,40
Aditivo Gamma,1.0000,1.0000,0.9824,40
Aditivo Uniforme,0.9615,0.1911,0.6387,40
Multiplicativo Gaussiano,0.0527,0.1012,0.7361,20
Multiplicativo Exponencial,0.0107,0.0133,0.0478,20
Multiplicativo Gamma,0.0006,0.0036,0.0097,20
Multiplicativo Uniforme,0.0220,0.0319,0.6108,20


## N = 100

### do(Y) = 0.0

In [14]:
resumen, p_values = resultados[(100, "ydo0")][0], resultados[(100, "ydo0")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.800000,0.930230,1.076530,0.064160,0.065060,0.563750,0.552500,Beta importada de Resumen_Observacional (N=100)
1,Aditivo Exponencial,0.400000,0.915990,0.869250,0.109420,0.085700,0.656250,0.660000,Beta importada de Resumen_Observacional (N=100)
2,Aditivo Gamma,0.600000,0.918570,1.072610,0.196480,0.196120,0.722500,0.760000,Beta importada de Resumen_Observacional (N=100)
3,Aditivo Uniforme,0.500000,0.946440,0.979190,0.050640,0.057400,0.603750,0.616250,Beta importada de Resumen_Observacional (N=100)
4,Multiplicativo Gaussiano,0.200000,0.945350,0.929550,0.104540,0.088950,0.710000,0.692500,Beta importada de Resumen_Observacional (N=100)
5,Multiplicativo Exponencial,0.600000,0.752200,0.724360,0.198910,0.141040,0.690000,0.690000,Beta importada de Resumen_Observacional (N=100)
6,Multiplicativo Gamma,0.300000,0.714910,0.653450,0.168520,0.100280,0.687500,0.650000,Beta importada de Resumen_Observacional (N=100)
7,Multiplicativo Uniforme,0.700000,0.905090,0.916310,0.084290,0.099580,0.627500,0.610000,Beta importada de Resumen_Observacional (N=100)


In [15]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.9996,0.6079,0.2677,20
Aditivo Exponencial,0.1471,0.0042,0.6032,20
Aditivo Gamma,0.9867,0.3506,0.9854,20
Aditivo Uniforme,0.9953,0.7955,0.8372,20
Multiplicativo Gaussiano,0.2461,0.0654,0.1172,10
Multiplicativo Exponencial,0.0186,0.0186,0.5469,10
Multiplicativo Gamma,0.0420,0.2158,0.3438,10
Multiplicativo Uniforme,0.9971,0.9033,0.1797,10


### do(Y) = 1.0

In [16]:
resumen, p_values = resultados[(100, "ydo1")][0], resultados[(100, "ydo1")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.800000,0.908590,0.977920,0.039970,0.039020,0.567500,0.552500,Beta importada de Resumen_Observacional (N=100)
1,Aditivo Exponencial,0.400000,0.833220,0.822790,0.091240,0.063250,0.646250,0.667500,Beta importada de Resumen_Observacional (N=100)
2,Aditivo Gamma,0.600000,0.790090,0.883510,0.128340,0.107990,0.692500,0.693750,Beta importada de Resumen_Observacional (N=100)
3,Aditivo Uniforme,0.500000,0.902680,0.909620,0.034410,0.039480,0.580000,0.572500,Beta importada de Resumen_Observacional (N=100)
4,Multiplicativo Gaussiano,0.200000,0.550600,0.544330,0.142470,0.128900,0.642500,0.642500,Beta importada de Resumen_Observacional (N=100)
5,Multiplicativo Exponencial,0.600000,0.477380,0.470560,0.312820,0.208150,0.725000,0.745000,Beta importada de Resumen_Observacional (N=100)
6,Multiplicativo Gamma,0.300000,0.461700,0.384700,0.260980,0.108760,0.725000,0.710000,Beta importada de Resumen_Observacional (N=100)
7,Multiplicativo Uniforme,0.700000,0.511250,0.529830,0.116390,0.155970,0.547500,0.555000,Beta importada de Resumen_Observacional (N=100)


In [17]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.9984,0.8695,0.0486,20
Aditivo Exponencial,0.2979,0.0000,0.9561,20
Aditivo Gamma,0.9982,0.1305,0.4038,20
Aditivo Uniforme,0.9940,0.9709,0.4176,20
Multiplicativo Gaussiano,0.6523,0.5771,0.4727,10
Multiplicativo Exponencial,0.3848,0.0098,0.9023,10
Multiplicativo Gamma,0.1162,0.0654,0.3789,10
Multiplicativo Uniforme,0.9580,0.8838,0.6445,10


### Media entre do(Y)=0.0 y do(Y)=1.0

In [18]:
resumen, p_values = resultados[(100, "media")][0], resultados[(100, "media")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.800000,0.919410,1.027220,0.052070,0.052040,0.565630,0.552500,Beta importada de Resumen_Observacional (N=100)
1,Aditivo Exponencial,0.400000,0.874600,0.846020,0.100330,0.074480,0.651250,0.663750,Beta importada de Resumen_Observacional (N=100)
2,Aditivo Gamma,0.600000,0.854330,0.978060,0.162410,0.152050,0.707500,0.726880,Beta importada de Resumen_Observacional (N=100)
3,Aditivo Uniforme,0.500000,0.924560,0.944410,0.042520,0.048440,0.591880,0.594380,Beta importada de Resumen_Observacional (N=100)
4,Multiplicativo Gaussiano,0.200000,0.747980,0.736940,0.123500,0.108920,0.676250,0.667500,Beta importada de Resumen_Observacional (N=100)
5,Multiplicativo Exponencial,0.600000,0.614790,0.597460,0.255860,0.174600,0.707500,0.717500,Beta importada de Resumen_Observacional (N=100)
6,Multiplicativo Gamma,0.300000,0.588310,0.519070,0.214750,0.104520,0.706250,0.680000,Beta importada de Resumen_Observacional (N=100)
7,Multiplicativo Uniforme,0.700000,0.708170,0.723070,0.100340,0.127770,0.587500,0.582500,Beta importada de Resumen_Observacional (N=100)


In [19]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,1.0000,0.7364,0.0719,40
Aditivo Exponencial,0.0984,0.0000,0.9154,40
Aditivo Gamma,0.9998,0.1324,0.9266,40
Aditivo Uniforme,0.9996,0.9647,0.7043,40
Multiplicativo Gaussiano,0.3371,0.1942,0.2200,20
Multiplicativo Exponencial,0.0715,0.0006,0.8504,20
Multiplicativo Gamma,0.0164,0.0413,0.3154,20
Multiplicativo Uniforme,0.9984,0.9552,0.3555,20


## Comparación entre las 6 combinaciones (N x do(Y))

Beta óptima elegida y número de métricas significativas (de 3: `MAE Z`, `HSIC(Z,X)`, `RF Acc`) en
cada combinación, para ver de un vistazo qué cambia con N y con la intervención.


In [20]:
combos = [(50, "ydo0"), (50, "ydo1"), (50, "media"), (100, "ydo0"), (100, "ydo1"), (100, "media")]

comparacion = pd.DataFrame({"Experimento": list(EXPERIMENTOS.keys())})
for n_filter, etiqueta in combos:
    resumen_c, p_values_c = resultados[(n_filter, etiqueta)]
    col_beta = f"beta_optima N={n_filter} {etiqueta}"
    col_sig = f"metricas_sig N={n_filter} {etiqueta}"
    comparacion = comparacion.merge(
        resumen_c[["Experimento", "beta_optima"]].rename(columns={"beta_optima": col_beta}),
        on="Experimento",
    )
    comparacion[col_sig] = comparacion["Experimento"].map((p_values_c[METRICAS] < ALPHA).sum(axis=1))

comparacion


,Experimento,beta_optima N=50 ydo0,metricas_sig N=50 ydo0,beta_optima N=50 ydo1,metricas_sig N=50 ydo1,beta_optima N=50 media,metricas_sig N=50 media,beta_optima N=100 ydo0,metricas_sig N=100 ydo0,beta_optima N=100 ydo1,metricas_sig N=100 ydo1,beta_optima N=100 media,metricas_sig N=100 media
0,Aditivo Gaussiano,0.9,0,0.9,0,0.9,0,0.8,0,0.8,1,0.8,0
1,Aditivo Exponencial,0.9,1,0.9,1,0.9,1,0.4,1,0.4,1,0.4,1
2,Aditivo Gamma,0.9,0,0.9,0,0.9,0,0.6,0,0.6,0,0.6,0
3,Aditivo Uniforme,0.3,0,0.3,0,0.3,0,0.5,0,0.5,0,0.5,0
4,Multiplicativo Gaussiano,0.4,1,0.4,0,0.4,0,0.2,0,0.2,0,0.2,0
5,Multiplicativo Exponencial,0.1,0,0.1,1,0.1,3,0.6,2,0.6,1,0.6,1
6,Multiplicativo Gamma,0.6,2,0.6,2,0.6,3,0.3,1,0.3,0,0.3,2
7,Multiplicativo Uniforme,0.4,2,0.4,0,0.4,2,0.7,0,0.7,0,0.7,0


## Guardar CSVs resumen

In [21]:
for (n_filter, etiqueta), (resumen_c, _) in resultados.items():
    out_path = TABLAS_DIR / f"resumen_intervencional_beta_n{n_filter}_{etiqueta}.csv"
    resumen_c.to_csv(out_path, index=False)
    print(f"Guardado en: {out_path}")

Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_intervencional_beta_n50_ydo0.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_intervencional_beta_n50_ydo1.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_intervencional_beta_n50_media.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_intervencional_beta_n100_ydo0.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_intervencional_beta_n100_ydo1.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_intervencional_beta_n100_media.csv
